⚡ Intermédiaire | ⏱ 45 min | 🔑 `_protégé`, `__privé`, `@property`, setter, invariant, copie

# Exercices - Encapsulation

Cours associé : [03-encapsulation.ipynb](03-encapsulation.ipynb).

## Comment travailler

Chaque exercice reproduit **un geste montré dans le cours**, dans un cas très proche. Pour chacun :

1. Lis l'**objectif** et les **étapes**.
2. Dans la cellule de code, remplace chaque `...` par ton code. Les noms des classes, méthodes et variables sont déjà écrits : garde-les, la vérification en a besoin.
3. Exécute ta cellule (`Shift + Entrée`) et compare avec le **résultat attendu**.
4. Exécute la cellule **Vérification** juste en dessous. Si elle affiche `OK`, c'est bon. Sinon, le message te dit ce qui ne va pas.

Si tu bloques, la ligne 📖 te dit quelle partie du cours relire.


## 1. L'idée

### 1.1 Cacher l'attribut, valider dans la méthode

**Objectif** : Empêcher un état absurde en passant par une méthode.

**À faire** :
1. Dans `__init__`, range la valeur dans `_quantite` (attribut interne).
2. `retirer(n)` lève `ValueError("retrait impossible")` si `n <= 0` ou si `n` dépasse la quantité ; sinon elle soustrait `n`.
3. `quantite()` renvoie la valeur interne.

**Résultat attendu** :
```
7
retrait impossible
```

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « 1. L'idée », cellule « Avec encapsulation : l'attribut devient interne, la méthode valide ».


In [ ]:
class Stock:
    def __init__(self, quantite):
        ...        # range quantite dans self._quantite (attribut interne)

    def retirer(self, n):
        ...        # si n <= 0 ou n > self._quantite : raise ValueError("retrait impossible")
        ...        # sinon soustrais n à self._quantite

    def quantite(self):
        ...        # renvoie la valeur interne

s = Stock(10)
s.retirer(3)
print(s.quantite())
try:
    s.retirer(50)
except ValueError as e:
    print(e)


In [ ]:
# Vérification
assert s.quantite() == 7, f"quantite doit valoir 7, obtenu {s.quantite()}"
assert "_quantite" in vars(s) and "quantite" not in vars(s), "la donnée doit être stockée dans _quantite"
for mauvais in (0, -1, 8):
    try:
        s.retirer(mauvais)
    except ValueError:
        pass
    else:
        raise AssertionError(f"retirer({mauvais}) devrait lever ValueError")
assert s.quantite() == 7, "un retrait refusé ne doit rien changer"
print("OK")


## 2. Visibilité

### 2.1 Prédire le name mangling

**Objectif** : Savoir sous quel nom Python range un attribut `__privé`.

**À faire** :
1. Lis le code sans l'exécuter.
2. Remplace chaque `...` de `prediction` par `True` ou `False`.

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « 2. Visibilité », cellule « __attribut : renommé par Python ».


In [ ]:
class Carte:
    def __init__(self):
        self.numero = "4970"
        self._solde = 50
        self.__pin = "1234"

carte = Carte()

prediction = {
    "hasattr(carte, 'numero')": ...,        # attribut public
    "hasattr(carte, '_solde')": ...,        # un seul _ : simple convention
    "hasattr(carte, '__pin')": ...,         # deux _ : Python renomme l'attribut
    "hasattr(carte, '_Carte__pin')": ...,   # le nom réel après renommage
}


In [ ]:
# Vérification
reel = {
    "hasattr(carte, 'numero')": hasattr(carte, "numero"),
    "hasattr(carte, '_solde')": hasattr(carte, "_solde"),
    "hasattr(carte, '__pin')": hasattr(carte, "__pin"),
    "hasattr(carte, '_Carte__pin')": hasattr(carte, "_Carte__pin"),
}
for cle, valeur in reel.items():
    assert prediction[cle] == valeur, f"{cle} : tu as prédit {prediction[cle]!r}, Python donne {valeur!r}"
print("OK")


### 2.2 Méthode interne

**Objectif** : Factoriser un traitement commun dans une méthode `_interne`.

**À faire** :
1. `_enregistrer(libelle, montant)` ajoute le tuple `(libelle, montant)` à `_historique`.
2. `deposer` et `retirer` modifient `_solde` puis appellent `_enregistrer` avec `"dépôt"` ou `"retrait"`.

**Résultat attendu** :
```
[('dépôt', 50), ('retrait', 20)] 130
```

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « 2. Visibilité », cellule « Une méthode interne commence aussi par _ ».


In [ ]:
class Compte:
    def __init__(self, solde):
        self._solde = solde
        self._historique = []

    def _enregistrer(self, libelle, montant):
        ...        # ajoute le tuple (libelle, montant) à self._historique

    def deposer(self, montant):
        ...        # ajoute montant à self._solde
        ...        # enregistre "dépôt"

    def retirer(self, montant):
        ...        # retire montant de self._solde
        ...        # enregistre "retrait"

c = Compte(100)
c.deposer(50)
c.retirer(20)
print(c._historique, c._solde)


In [ ]:
# Vérification
assert c._historique == [("dépôt", 50), ("retrait", 20)], f"historique inattendu : {c._historique}"
assert c._solde == 130, f"solde attendu 130, obtenu {c._solde}"
print("OK")


## 3. `@property`

### 3.1 Lecture seule

**Objectif** : Exposer une donnée sans permettre de la modifier.

**À faire** :
1. Range le paramètre dans `_numero`.
2. Déclare une property `numero` qui le renvoie, sans setter.

**Résultat attendu** :
```
FR76
property 'numero' of 'Compte' object has no setter
```

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « 3. `@property` », cellule « @property : lecture via c.solde ».


In [ ]:
class Compte:
    def __init__(self, numero):
        ...        # range numero dans self._numero

    ...            # décorateur qui transforme la méthode en attribut en lecture
    def numero(self):
        ...        # renvoie self._numero

c = Compte("FR76")
print(c.numero)
try:
    c.numero = "XX"
except AttributeError as e:
    print(e)


In [ ]:
# Vérification
assert c.numero == "FR76", f"numero doit valoir 'FR76', obtenu {c.numero!r}"
assert isinstance(vars(Compte)["numero"], property), "numero doit être une property"
try:
    c.numero = "XX"
except AttributeError:
    pass
else:
    raise AssertionError("numero ne doit pas être modifiable")
print("OK")


### 3.2 Setter avec validation

**Objectif** : Refuser une valeur invalide à l'écriture.

**À faire** :
1. Dans `__init__`, écris `self.prix = prix` (pas `_prix`) pour passer par le setter.
2. Le getter renvoie `_prix`.
3. Le setter lève `ValueError("prix invalide")` si la valeur est `<= 0`, sinon il stocke dans `_prix`.

**Résultat attendu** :
```
19.9
prix invalide
```

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « 3. `@property` », cellule « @solde.setter : l'écriture passe par une validation ».


In [ ]:
class Produit:
    def __init__(self, nom, prix):
        self.nom = nom
        ...        # self.prix = prix : passe par le setter

    @property
    def prix(self):
        ...        # renvoie self._prix

    ...            # décorateur setter de prix
    def prix(self, valeur):
        ...        # si valeur <= 0 : raise ValueError("prix invalide")
        ...        # sinon stocke dans self._prix

p = Produit("Livre", 19.9)
print(p.prix)
try:
    p.prix = -5
except ValueError as e:
    print(e)


In [ ]:
# Vérification
assert p.prix == 19.9, f"prix doit rester 19.9 après le refus, obtenu {p.prix}"
p.prix = 25
assert p.prix == 25 and p._prix == 25, "le setter doit stocker dans _prix"
try:
    Produit("Gratuit", 0)
except ValueError:
    pass
else:
    raise AssertionError("un prix de 0 doit être refusé dès le constructeur")
print("OK")


### 3.3 Propriété calculée

**Objectif** : Exposer une valeur dérivée, jamais stockée.

**À faire** :
1. `aire` renvoie `largeur * hauteur`.
2. `perimetre` renvoie `2 * (largeur + hauteur)`.
3. Les deux sont des properties : on les lit sans parenthèses.

**Résultat attendu** :
```
6 10
30 26
```

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « 3. `@property` », cellule « Propriété calculée ».


In [ ]:
class Rectangle:
    def __init__(self, largeur, hauteur):
        self.largeur = largeur
        self.hauteur = hauteur

    ...            # décorateur property
    def aire(self):
        ...        # largeur * hauteur

    ...            # décorateur property
    def perimetre(self):
        ...        # 2 * (largeur + hauteur)

r = Rectangle(2, 3)
print(r.aire, r.perimetre)
r.largeur = 10
print(r.aire, r.perimetre)


In [ ]:
# Vérification
assert (r.aire, r.perimetre) == (30, 26), f"obtenu {(r.aire, r.perimetre)}"
assert "aire" not in vars(r), "aire ne doit pas être stockée, mais calculée"
assert isinstance(vars(Rectangle)["perimetre"], property), "perimetre doit être une property"
print("OK")


### 3.4 Deux vues, une donnée

**Objectif** : Stocker une seule valeur et l'exposer dans deux unités.

**À faire** :
1. Seuls les mètres sont stockés, dans `_metres`.
2. `kilometres` (getter) renvoie `_metres / 1000`.
3. `kilometres` (setter) convertit la valeur reçue en mètres et la stocke.

**Résultat attendu** :
```
2.5 2500
```

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « 3. `@property` », cellule « Deux propriétés, une seule donnée stockée ».


In [ ]:
class Distance:
    def __init__(self, metres):
        self._metres = metres

    @property
    def metres(self):
        return self._metres

    @property
    def kilometres(self):
        ...        # self._metres / 1000

    @kilometres.setter
    def kilometres(self, valeur):
        ...        # convertit valeur en mètres et stocke dans self._metres

d = Distance(1500)
d.kilometres = 2.5
print(d.kilometres, d.metres)


In [ ]:
# Vérification
assert d.metres == 2500, f"metres doit valoir 2500, obtenu {d.metres}"
assert d.kilometres == 2.5
assert set(vars(d)) == {"_metres"}, f"seule _metres doit être stockée, trouvé {set(vars(d))}"
print("OK")


## 4. Invariants

### 4.1 Valeur bornée

**Objectif** : Garantir `0 <= niveau <= 100` quoi qu'il arrive.

**À faire** :
1. Le setter de `niveau` borne la valeur entre 0 et 100 avec `max` et `min`.
2. `remplir(n)` ajoute `n` au niveau, `vider(n)` retire `n` : les deux passent par le setter (`self.niveau = ...`).

**Résultat attendu** :
```
100 0
```

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « 4. Invariants », cellule « Invariant 0 <= batterie <= 100 ».


In [ ]:
class Jauge:
    def __init__(self, niveau=50):
        self.niveau = niveau

    @property
    def niveau(self):
        return self._niveau

    @niveau.setter
    def niveau(self, valeur):
        ...        # borne valeur entre 0 et 100 avec max() et min(), stocke dans self._niveau

    def remplir(self, n):
        ...        # self.niveau += n : passe par le setter

    def vider(self, n):
        ...        # self.niveau -= n : passe par le setter

j = Jauge()
j.remplir(80)
a = j.niveau
j.vider(500)
print(a, j.niveau)


In [ ]:
# Vérification
assert (a, j.niveau) == (100, 0), f"obtenu {(a, j.niveau)}"
assert Jauge(250).niveau == 100 and Jauge(-3).niveau == 0, "le constructeur doit aussi borner"
k = Jauge(10)
k.remplir(15)
assert k.niveau == 25
print("OK")


## 5. Ne pas laisser fuir l'état

### 5.1 Renvoyer une copie

**Objectif** : Exposer une collection interne sans permettre de la modifier.

**À faire** :
1. `livres` est une property qui renvoie un **tuple** construit à partir de `_livres`.

**Résultat attendu** :
```
('Dune', '1984') 2
```

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « 5. Ne pas laisser fuir l'état interne », cellule « Correction : renvoyer un tuple ».


In [ ]:
class Bibliotheque:
    def __init__(self):
        self._livres = []

    def ajouter(self, titre):
        self._livres.append(titre)

    ...            # décorateur property
    def livres(self):
        ...        # renvoie un tuple construit à partir de self._livres

b = Bibliotheque()
b.ajouter("Dune")
b.ajouter("1984")
copie = list(b.livres)
copie.clear()
print(b.livres, len(b.livres))


In [ ]:
# Vérification
assert isinstance(b.livres, tuple), f"livres doit renvoyer un tuple, obtenu {type(b.livres).__name__}"
assert b.livres == ("Dune", "1984"), f"obtenu {b.livres}"
assert b.livres is not b._livres
print("OK")


## 6. Couplage

### 6.1 Utiliser l'interface de l'autre classe

**Objectif** : Écrire une classe qui n'accède jamais aux attributs `_` d'une autre.

**À faire** :
1. `Maison.chauffer(thermostat, temperature)` doit régler le thermostat en appelant sa méthode publique `regler`, pas en touchant `_consigne`.
2. La vérification utilise un faux thermostat qui n'a **pas** d'attribut `_consigne` : si tu y accèdes, elle échoue.

**Résultat attendu** :
```
21
```

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « 6. Parler à l'interface, pas aux entrailles », cellule « ✅ Couplage faible ».


In [ ]:
class Thermostat:
    def __init__(self):
        self._consigne = 19

    def regler(self, temperature):
        self._consigne = max(5, min(30, temperature))

    @property
    def consigne(self):
        return self._consigne

class Maison:
    def chauffer(self, thermostat, temperature):
        ...        # appelle la méthode publique regler() du thermostat, jamais _consigne

t = Thermostat()
Maison().chauffer(t, 21)
print(t.consigne)


In [ ]:
# Vérification
assert t.consigne == 21, f"consigne attendue 21, obtenu {t.consigne}"

class FauxThermostat:
    def __init__(self):
        self.appels = []

    def regler(self, temperature):
        self.appels.append(temperature)

faux = FauxThermostat()
Maison().chauffer(faux, 18)
assert faux.appels == [18], "chauffer doit passer par la méthode regler()"
assert not hasattr(faux, "_consigne"), "chauffer ne doit pas créer ni toucher _consigne"
print("OK")


## 7. Pièges

### 7.1 Corriger la récursion du getter

**Objectif** : Repérer un getter qui s'appelle lui-même.

**À faire** :
1. Le code ci-dessous lève `RecursionError`. Trouve la ligne fautive dans le getter et corrige-la.

**Résultat attendu** :
```
100
```

📖 Cours [03-encapsulation.ipynb](03-encapsulation.ipynb), section « ⚠️ Pièges », cellule « BUG classique : le getter se rappelle lui-même ».


In [ ]:
# Un bug à corriger : le getter lève RecursionError
class Compte:
    def __init__(self, solde):
        self._solde = solde

    @property
    def solde(self):
        return self.solde

c = Compte(100)
print(c.solde)


In [ ]:
# Vérification
assert c.solde == 100, f"solde attendu 100, obtenu {c.solde}"
assert isinstance(vars(Compte)["solde"], property), "solde doit rester une property"
print("OK")


## Bravo

Tu as reproduit tous les gestes du cours. Suite : [05-heritage-polymorphisme-composition.ipynb](05-heritage-polymorphisme-composition.ipynb).
